### PySpark Otomoto Demo 

Źródło danych: https://www.kaggle.com/datasets/szymoncyperski/car-sales-offers-from-otomotopl-2023 


In [ ]:
import os
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import matplotlib.pyplot as plt

**Teoria:** Powyżej importujemy niezbędne biblioteki. `SparkSession` to główny punkt wejścia do funkcjonalności DataFrame i SQL w Sparku (od wersji 2.0). Moduł `functions` dostarcza wbudowane funkcje operujące na kolumnach, a `matplotlib.pyplot` posłuży nam do późniejszej wizualizacji danych.


In [ ]:
spark = SparkSession.builder \
    .appName("Otomoto Demo") \
    .getOrCreate()


**Teoria:** Tworzymy sesję Sparka. `builder` używa wzorca projektowego Builder do skonfigurowania sesji. `getOrCreate()` tworzy nową sesję lub pobiera istniejącą, co jest bezpieczne przy wielokrotnym uruchamianiu notatnika.


In [ ]:
df = spark.read.option("header", True) \
    .option("delimiter", ";") \
    .option("inferSchema", False) \
    .csv("otomoto_offers_eng_23-04-2023.csv")


**Teoria:** Wczytywanie danych. Spark używa leniwego ewaluowania (lazy evaluation) - dane nie są fizycznie wczytywane w tym momencie, tworzony jest tylko plan wykonania (DAG). Ustawiamy `header=True` ponieważ nasz plik CSV ma nagłówki, oraz określamy separator jako średnik `;`.


In [ ]:
df.show()

**Teoria:** `show()` to akcja (action), która uruchamia wykonanie obliczeń w Sparku. Dopiero teraz plik jest odczytywany, a wynik prezentowany na ekranie.


In [ ]:
df.filter(F.col("vehicle_brand") == "Volvo").show()

In [ ]:
df = df.withColumn("price_num",
                   F.regexp_replace(F.col("price"), r"[^\d]", "").cast("double"))

df = df.withColumn("mileage_km",
                   F.regexp_replace(F.col("mileage"), r"[^\d]", "").cast("integer"))

df = df.withColumn("production_year_int",
                   F.regexp_replace(F.col("production_year"), r"[^\d]", "").cast("integer"))

df = df.withColumn("engine_cc",
                   F.regexp_replace(F.col("engine_displacement"), r"[^\d]", "").cast("integer"))

df = df.withColumn("power_hp",
                   F.regexp_replace(F.col("power"), r"[^\d]", "").cast("integer"))

df = df.withColumn("fuel_clean",
                   F.lower(F.trim(F.col("fuel_type"))))

In [ ]:
df.select("vehicle_brand", "vehicle_model", "price_num", "mileage_km",
          "production_year_int", "engine_cc", "power_hp", "fuel_clean") \
  .show(10, truncate=False)

**Teoria:** `select()` to transformacja, która działa jak w SQL - pozwala wybrać podzbiór kolumn. Zmniejsza to ilość przetwarzanych danych w dalszych krokach.


In [ ]:
avg_brand = df.groupBy("vehicle_brand") \
              .agg(F.round(F.avg("price_num"), 2).alias("avg_price")) \
              .orderBy(F.col("avg_price").desc())

print("Średnia cena per marka")
avg_brand.show(20, truncate=False)

In [ ]:
fuel_count = df.groupBy("fuel_clean").count()
print("Liczba ogłoszeń wg rodzaju paliwa")
fuel_count.show()

In [ ]:
df.createOrReplaceTempView("cars")

In [ ]:
# SQL: zależność mocy i pojemności od ceny
spark.sql("""
    SELECT vehicle_brand,
           ROUND(AVG(power_hp), 1) AS avg_power,
           ROUND(AVG(engine_cc), 1) AS avg_cc,
           ROUND(AVG(price_num), 1) AS avg_price
    FROM cars
    GROUP BY vehicle_brand
    ORDER BY avg_power DESC
""").show()

In [ ]:
df.groupBy("production_year_int") \
  .count() \
  .orderBy(F.col("production_year_int").desc()) \
  .show()

In [ ]:
# Średnia cena i przebieg per marka i model
df.groupBy("vehicle_brand", "vehicle_model") \
  .agg(
      F.round(F.avg("price_num"), 2).alias("avg_price"),
      F.round(F.avg("mileage_km"), 2).alias("avg_mileage")
  ) \
  .orderBy(F.col("avg_price").desc()) \
  .show(20, truncate=False)

In [ ]:
# zależność ceny od przebiegu 
price_mileage = df.select("price_num", "mileage_km") \
                  .where((F.col("price_num").isNotNull()) & (F.col("mileage_km").isNotNull()))

In [ ]:
pdf_scatter = price_mileage.sample(fraction=0.1, seed=42).toPandas()

plt.figure(figsize=(8,5))
plt.scatter(pdf_scatter["mileage_km"], pdf_scatter["price_num"], s=6)
plt.title("Cena vs Przebieg")
plt.xlabel("Przebieg [km]")
plt.ylabel("Cena")
plt.tight_layout()
plt.savefig("scatter_price_mileage.png")

print("Wizualizacja scatter zapisana jako scatter_price_mileage.png")

**Teoria:** `toPandas()` to akcja, która zbiera (collect) wszystkie dane na partycjach roboczych i przesyła je na węzeł główny (Driver), konwertując do struktury Pandas DataFrame. Uwaga: Można tego używać tylko na małych zbiorach (po limitowaniu np. top 10), w przeciwnym razie braknie pamięci RAM na Driverze!


---
# Zadanie samodzielne: Analiza Przestępczości w Chicago

Poniżej znajduje się miejsce na realizację zadania z analizy danych przy użyciu PySpark na zbiorze *Chicago Crimes* (około 50 000 ostatnich zdarzeń). Twoim celem jest przygotowanie, wyczyszczenie oraz zanalizowanie tych danych z wykorzystaniem zaawansowanych optymalizacji dostępnych w Sparku.

### Wymagania:
1. **Wczytanie i Czyszczenie Danych:** Wczytaj pobrany plik `chicago_crimes_sample.csv`. Usuń ewentualne duplikaty, wiersze z brakami danych (szczególnie w kluczowych kolumnach) i odfiltruj/napraw błędne daty.
2. **UDF i Pora Dnia:** Dodaj nową kolumnę z klasyfikacją pory dnia (np. noc, dzień, wieczór) utworzoną za pomocą User Defined Function (UDF) w oparciu o godzinę z kolumny `Date`.
3. **Optymalizacja i Partycjonowanie:** Zoptymalizuj przetwarzanie. Zastanów się, w których momentach użyć `cache()`. Przy dołączaniu mniejszych tabel słownikowych (jeśli byś je tworzył), wykorzystaj *broadcast join*. Ostatecznie zapisz przefiltrowane dane do formatu **Parquet** z podziałem na partycje według roku (`Year`).
4. **Analiza i Plany Zapytań:** Przeprowadź analizę statystyczną przestępstw (np. jakiego typu przestępstwa są najpopularniejsze w konkretnych lokacjach, o konkretnym czasie). Wykorzystaj funkcję `.explain()` aby udokumentować plan zapytania Sparka dla najcięższej agregacji.
5. *(Opcjonalnie)* **Uczenie Maszynowe (MLlib):** Spróbuj zbudować i wytrenować prosty model wieloklasowy, przewidujący rodzaj przestępstwa (`Primary Type`) na podstawie innych atrybutów, jak lokacja, godzina, arrest itp.

In [7]:
# Tutaj wpisz swój kod zliczający, czytający plik itp.
import os
import sys
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, udf, to_timestamp, hour, broadcast, count
from pyspark.sql.types import StringType
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline

spark = SparkSession.builder \
    .appName("Chicago Crimes Analysis") \
    .getOrCreate()

# Podpowiedź krok 1:
df_crimes = spark.read.option("header", True).csv("chicago_crimes_sample.csv")
df_clean = df_crimes.dropDuplicates()

#braki danych w kluczowych kolumnach
key_columns = ["date", "primary_type", "location_description", "year"]
df_clean = df_clean.dropna(subset=key_columns)

# błędne daty
df_clean = df_clean.withColumn("Parsed_Date", to_timestamp(col("date"), "yyyy-MM-dd'T'HH:mm:ss.SSS"))
df_clean = df_clean.filter(col("Parsed_Date").isNotNull())
df_clean.show(5)

+--------+-----------+--------------------+------------------+----+------------+--------------+--------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+--------------------+------------+-------------+--------+-------------------+
|      id|case_number|                date|             block|iucr|primary_type|   description|location_description|arrest|domestic|beat|district|ward|community_area|fbi_code|x_coordinate|y_coordinate|year|          updated_on|    latitude|    longitude|location|        Parsed_Date|
+--------+-----------+--------------------+------------------+----+------------+--------------+--------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+--------------------+------------+-------------+--------+-------------------+
|14189600|   JK245352|2026-05-06T15:00:...|059XX S RACINE AVE|0820|       THEFT|$500 AND UNDER|         GAS STATION| false|   false|0713|     007|  

In [8]:
# UDF i Pora Dnia 
def classify_time_of_day(hr):
    if hr is None: return "None"
    if 6 <= hr < 12: return "Morning"
    elif 12 <= hr < 18: return "Afternoon"
    elif 18 <= hr < 22: return "Evening"
    else: return "Night"

time_of_day_udf = udf(classify_time_of_day, StringType())
df_with_time = df_clean.withColumn("Hour", hour(col("Parsed_Date"))) \
                       .withColumn("Time_Of_Day", time_of_day_udf(col("Hour")))

# Optymalizacja i Partycjonowanie
df_with_time.cache()
severity_data = [("THEFT", "Low"), ("BATTERY", "High"), ("HOMICIDE", "Critical"), ("NARCOTICS", "Medium")]
severity_df = spark.createDataFrame(severity_data, ["primary_type", "Severity"])
df_optimized = df_with_time.join(broadcast(severity_df), on="primary_type", how="left")
df_optimized = df_optimized.fillna({"Severity": "Unknown"})
df_optimized.write.mode("overwrite").partitionBy("year").parquet("output/chicago_crimes_partitioned.parquet")

# Analiza i plany zapytań
heavy_aggregation = df_optimized.groupBy("location_description", "Time_Of_Day", "primary_type") \
                                .agg(count("*").alias("Crime_Count")) \
                                .orderBy(col("Crime_Count").desc())
heavy_aggregation.show(truncate=False)
heavy_aggregation.explain(extended=True)

+--------------------+-----------+-------------------+-----------+
|location_description|Time_Of_Day|primary_type       |Crime_Count|
+--------------------+-----------+-------------------+-----------+
|STREET              |Night      |MOTOR VEHICLE THEFT|1068       |
|APARTMENT           |Night      |BATTERY            |1030       |
|STREET              |Night      |CRIMINAL DAMAGE    |873        |
|STREET              |Evening    |MOTOR VEHICLE THEFT|771        |
|APARTMENT           |Afternoon  |BATTERY            |744        |
|STREET              |Night      |THEFT              |728        |
|STREET              |Afternoon  |MOTOR VEHICLE THEFT|627        |
|APARTMENT           |Evening    |BATTERY            |625        |
|APARTMENT           |Morning    |BATTERY            |620        |
|SMALL RETAIL STORE  |Afternoon  |THEFT              |591        |
|APARTMENT           |Afternoon  |THEFT              |589        |
|STREET              |Afternoon  |THEFT              |556     

In [9]:
# Uczenie Maszynowe
ml_df = df_optimized.dropna(subset=["location_description", "Time_Of_Day", "arrest", "primary_type"])
ml_df = ml_df.withColumn("Arrest_Str", col("arrest").cast(StringType()))

indexer_loc = StringIndexer(inputCol="location_description", outputCol="Loc_Index", handleInvalid="keep")
indexer_time = StringIndexer(inputCol="Time_Of_Day", outputCol="Time_Index", handleInvalid="keep")
indexer_arrest = StringIndexer(inputCol="Arrest_Str", outputCol="Arrest_Index", handleInvalid="keep")
indexer_label = StringIndexer(inputCol="primary_type", outputCol="label", handleInvalid="skip")
assembler = VectorAssembler(
    inputCols=["Loc_Index", "Time_Index", "Arrest_Index"],
    outputCol="features"
)

rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=10, maxDepth=5, maxBins=150)
pipeline = Pipeline(stages=[
    indexer_loc, indexer_time, indexer_arrest, indexer_label, assembler, rf
])

train_data, test_data = ml_df.randomSplit([0.8, 0.2], seed=42)
model = pipeline.fit(train_data)
predictions = model.transform(test_data)
predictions.select("primary_type", "label", "prediction", "probability").show(10, truncate=False)

+------------+-----+----------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|primary_type|label|prediction|probability                                                                                                                                                                                                                                                                                                                                                   

In [10]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.sql.functions import expr

print("wyniki predykcji")
evaluator = MulticlassClassificationEvaluator(
    labelCol="label", 
    predictionCol="prediction", 
    metricName="accuracy"
)
accuracy = evaluator.evaluate(predictions)
print(f"dokladnosc: {accuracy * 100:.2f}%\n")

rf_model = model.stages[5]
importances = rf_model.featureImportances

print("Ważność cech przy przewidywaniu:")
print(f"Lokalizacja: {importances[0]:.4f}")
print(f"Czas dnia: {importances[1]:.4f}")
print(f"Aresztowanie: {importances[2]:.4f}\n")
results = predictions.withColumn("Trafienie", expr("label == prediction"))
results.groupBy("Trafienie").count().show()

wyniki predykcji
dokladnosc: 30.60%

Ważność cech przy przewidywaniu:
Lokalizacja: 0.7624
Czas dnia: 0.0450
Aresztowanie: 0.1925

+---------+-----+
|Trafienie|count|
+---------+-----+
|     true| 3069|
|    false| 6962|
+---------+-----+



Na podstawie analizy ważności cech widać, że lokalizacja była najważniejszym czynnikiem dla modelu - najmocniej determinuje rodzaj przestępstwa.
Natomiast fakt czy kogoś areszowano może pomóc w wskazaniu modelowi z jakim typem przestęptswa ma doczynienia.